In [1]:
import subprocess
import sys
from check_metrics import check_metrics
import pandas as pd
from rdkit.Chem import rdBase
rdBase.DisableLog('rdApp.error')
rdBase.DisableLog('rdApp.warning')
def run_command(command):
    """Runs a command in the shell and checks for errors."""
    print(f"\n--> Executing: {' '.join(command)}")
    try:
        # capture output to see what happens during execution
        subprocess.run(command, check=True)
        print("Command executed successfully.")
    except subprocess.CalledProcessError as e:
        print(f"ERROR: Command failed with exit code {e.returncode}")
        # You might want to break or continue here based on desired behavior
        sys.exit(1)
    except FileNotFoundError:
        print("ERROR: A required command (like python, do_metrics.py, etc.) was not found.")
        sys.exit(1)


def main():
    from glob import glob
    #dataDir = "log3d_6lu7_sigmoid"
    #dataDirs = glob("dude_50020_constraint_all/*.csv")
    tgts = ["akt1", "ampc", "cp3a4", "cxcr4", "gcr","hivpr","hivrt","kif11"]
    for tgt in tgts:
            # Construct the command for do_metrics.py
            df_metrics = pd.DataFrame(columns=["Validity", "Uniqueness", "Novelty", "Diversity", "QED","MolWt"])
            from glob import glob
            paths = glob(f"dude_50020_constraint_all/concat_{tgt}_*.csv")
            for i,path in enumerate(paths):
                df = pd.read_csv(path)
                required_columns = ["Validity", "Uniqueness", "Novelty", "Diversity", "QED","MolWt"]
                metrics = check_metrics(df, required_columns)
                df_metrics.loc[i] = metrics

            print(f"Metrics for {tgt}:")
            print(df_metrics.describe())
            """ for col, value in zip(required_columns, df_metrics):
                print(f"{col}: {value:.4f}")
         """
        
main()

Metrics for akt1:
       Validity  Uniqueness   Novelty  Diversity       QED       MolWt
count       5.0    5.000000  5.000000   5.000000  5.000000    5.000000
mean        1.0    0.922592  0.999757   0.783993  0.480520  301.232167
std         0.0    0.010296  0.000544   0.024239  0.042006   18.688266
min         1.0    0.905109  0.998783   0.751825  0.435264  280.583408
25%         1.0    0.923401  1.000000   0.764041  0.446590  289.381684
50%         1.0    0.924284  1.000000   0.798575  0.473565  294.304428
75%         1.0    0.928929  1.000000   0.802550  0.516507  319.370382
max         1.0    0.931238  1.000000   0.802972  0.530672  322.520934
Metrics for ampc:
       Validity  Uniqueness  Novelty  Diversity       QED       MolWt
count       5.0    5.000000      5.0   5.000000  5.000000    5.000000
mean        1.0    0.930370      1.0   0.809500  0.466959  253.819241
std         0.0    0.008296      0.0   0.015866  0.018683   14.609715
min         1.0    0.919298      1.0   0.7976

In [2]:
from glob import glob

initial_pic50 = {
    "akt1": 4.737,
    "ampc": 3.766,
    "cp3a4": 3.644,
    "cxcr4": 5.099,
    "gcr": 3.803,
    "hivpr": 4.536,
    "hivrt": 3.411,
    "kif11": 3.843,
}

results = []
for tgt, initial in initial_pic50.items():
    valid = 0
    above = 0
    for path in sorted(glob(f"dude_50020_constraint_all/concat_{tgt}_*.csv")):
        df = pd.read_csv(path)
        reboltz = pd.to_numeric(df["ReBoltz"], errors="coerce").dropna()
        generated_pic50 = 6.0 - reboltz
        valid += len(generated_pic50)
        above += (generated_pic50 > initial).sum()

    results.append(
        {
            "Target": tgt.upper(),
            "Initial pIC50": initial,
            "Above initial": above,
            "Valid generated": valid,
            "Percentage": 100 * above / valid,
        }
    )

results_df = pd.DataFrame(results)
overall_above = results_df["Above initial"].sum()
overall_valid = results_df["Valid generated"].sum()
results_df.loc[len(results_df)] = {
    "Target": "Overall",
    "Initial pIC50": None,
    "Above initial": overall_above,
    "Valid generated": overall_valid,
    "Percentage": 100 * overall_above / overall_valid,
}

print(results_df.to_string(index=False, formatters={"Percentage": "{:.1f}".format}))

latex_df = results_df.copy()
latex_df["Initial pIC50"] = latex_df["Initial pIC50"].map(
    lambda value: "--" if pd.isna(value) else f"{value:.3f}"
)
latex_df["Generated molecules"] = (
    latex_df["Above initial"].astype(int).astype(str)
    + "/"
    + latex_df["Valid generated"].astype(int).astype(str)
)
latex_df["Higher than initial (\\%)"] = latex_df["Percentage"].map(
    lambda value: f"{value:.1f}"
)
latex_df = latex_df[["Target", "Initial pIC50", "Generated molecules", "Higher than initial (\\%)"]]
print("\\nLaTeX table:\\n")
print(latex_df.to_latex(index=False, escape=False))

 Target  Initial pIC50  Above initial  Valid generated Percentage
   AKT1          4.737           3180             5110       62.2
   AMPC          3.766           6139             7112       86.3
  CP3A4          3.644           4810             5094       94.4
  CXCR4          5.099           2429             4000       60.7
    GCR          3.803           3582             4458       80.3
  HIVPR          4.536           2951             5277       55.9
  HIVRT          3.411           4951             5091       97.3
  KIF11          3.843           5501             6793       81.0
Overall            NaN          33543            42935       78.1
\nLaTeX table:\n
\begin{tabular}{llll}
\toprule
Target & Initial pIC50 & Generated molecules & Higher than initial (\%) \\
\midrule
AKT1 & 4.737 & 3180/5110 & 62.2 \\
AMPC & 3.766 & 6139/7112 & 86.3 \\
CP3A4 & 3.644 & 4810/5094 & 94.4 \\
CXCR4 & 5.099 & 2429/4000 & 60.7 \\
GCR & 3.803 & 3582/4458 & 80.3 \\
HIVPR & 4.536 & 2951/5277 & 55.9

/tmp/ipykernel_1864381/510194174.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df.loc[len(results_df)] = {
